# Fine Tuning XLRS to Japanese and US English


**Author:**

Salvador Wahnon Palma s2665070@u.tsukuba.ac.jp

University of Tsukuba / Interaction Lab


<br>

**Objective:**
 
Generates L1-Informed vs Articulatory corrective feedback for English speakers learning Japanese pronunciation.

---

# 1. Setup Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


# 2. Organize English Dataset

In [2]:
from datasets import load_dataset

ds = load_dataset("kylelovesllms/timit_asr_ipa")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/341M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [3]:
def ExamineDataset(ds):
    print("=== Split Sizes ===")
    print(f"Test: {len(ds['test'])}")
    print(f"Validation: {len(ds['validation'])}")
    print(f"Train: {len(ds['train'])}")

    print("\n=== Features ===")
    for feature in ds["train"].features:
        print(f"{feature}: {ds['train'].features[feature]}")

    print("\n=== Entry ===")
    print(ds["train"][0])

    print("\n=== Sample ===")
    audio_sample = ds["train"][1]["audio"]

    samples = audio_sample.get_all_samples()

    print(samples)
    print(samples.sample_rate)
    print(samples.data[0].dtype)
    print(samples.data.shape)
    
ExamineDataset(ds)


=== Split Sizes ===
Test: 670
Validation: 670
Train: 3629

=== Features ===
audio: Audio(sampling_rate=None, decode=True, stream_index=None)
phonetic_detail: List({'start': Value('int64'), 'stop': Value('int64'), 'utterance': Value('string')})
word_detail: List({'start': Value('int64'), 'stop': Value('int64'), 'utterance': Value('string')})
text: Value('string')
duration: Value('float64')
timit_path: Value('string')
dialect_region: Value('string')
dialect_region_name: Value('string')
speaker_id: Value('string')
speaker_sex: Value('string')
id: Value('string')
sentence_type: Value('string')
ipa_transcription: List(Value('string'))

=== Entry ===
{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7f0c70424470>, 'phonetic_detail': [{'start': 0, 'stop': 1960, 'utterance': 'h#'}, {'start': 1960, 'stop': 2466, 'utterance': 'w'}, {'start': 2466, 'stop': 3480, 'utterance': 'ix'}, {'start': 3480, 'stop': 4000, 'utterance': 'dcl'}, {'start': 4000, 'stop': 5960, 'utterance': 's'}, 

In [4]:
def JoinIPA(example):
    example["IPA"] = " ".join(example["ipa_transcription"])
    return example

new_ds = ds.map(JoinIPA)
new_ds = new_ds.select_columns(["audio", "IPA"])

ExamineDataset(new_ds)

Map:   0%|          | 0/3629 [00:00<?, ? examples/s]

Map:   0%|          | 0/670 [00:00<?, ? examples/s]

Map:   0%|          | 0/670 [00:00<?, ? examples/s]

=== Split Sizes ===
Test: 670
Validation: 670
Train: 3629

=== Features ===
audio: Audio(sampling_rate=None, decode=True, stream_index=None)
IPA: Value('string')

=== Entry ===
{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7f0a4bd71040>, 'IPA': 'w ɪ d s ʌ t tʃ ɪ n æ k t ɪ v r ɪ f j u ʒ l b i j u s f l'}

=== Sample ===
AudioSamples:
  data (shape): torch.Size([1, 40448])
  pts_seconds: 0.0
  duration_seconds: 2.528
  sample_rate: 16000

16000
torch.float32
torch.Size([1, 40448])


In [5]:
for split, dataset in new_ds.items():
    dataset.to_parquet(f"/content/drive/MyDrive/Tsukuba/Datasets and Models/US_{split}.parquet")

Creating parquet from Arrow format:   0%|          | 0/37 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

# 3. Organize Japanese Dataset (local)

In [3]:
import os

JVS_DIR = "../Data/JVS/"

folders = [f for f in os.listdir(JVS_DIR) if os.path.isdir(os.path.join(JVS_DIR, f))] if os.path.isdir(JVS_DIR) else []

print(f"{len(folders)} Folders found:")
print(folders)
print("\n")

folders = [f"{folder}/parallel100" for folder in folders]

sample_folder = folders[0]

print(f"Sample folder: {sample_folder}")

wavFiles = [f for f in os.listdir(os.path.join(JVS_DIR, f"{sample_folder}/wav24kHz16bit")) if f.endswith(".wav")]
labFiles = [f for f in os.listdir(os.path.join(JVS_DIR, f"{sample_folder}/lab/mon")) if f.endswith(".lab")]


print(f"Audio files found: {len(wavFiles)}")
print(wavFiles)
print(f"Label files found: {len(labFiles)}")
print(labFiles)
print("\n")


def GetWavFile(speakerID, utteranceID):
    assert speakerID < 100
    assert utteranceID < 100
    
    folders = [f for f in os.listdir(JVS_DIR) if os.path.isdir(os.path.join(JVS_DIR, f))] if os.path.isdir(JVS_DIR) else []
    folders = [f"{folder}/parallel100" for folder in folders]
    sample_folder = folders[speakerID]
    try:
        result_file = [f for f in os.listdir(os.path.join(JVS_DIR, f"{sample_folder}/wav24kHz16bit")) if f.endswith(".wav")][utteranceID]
    except IndexError:
        raise ValueError(f"No .wav file found for speakerID {speakerID} and utteranceID {utteranceID}.")
    return os.path.join(JVS_DIR, f"{sample_folder}/wav24kHz16bit/{result_file}")

def GetLabFile(speakerID, utteranceID):
    assert speakerID < 100
    assert utteranceID < 100
    folders = [f for f in os.listdir(JVS_DIR) if os.path.isdir(os.path.join(JVS_DIR, f))] if os.path.isdir(JVS_DIR) else []
    folders = [f"{folder}/parallel100" for folder in folders]
    sample_folder = folders[speakerID]
    
    try: 
        result_file = [f for f in os.listdir(os.path.join(JVS_DIR, f"{sample_folder}/lab/mon")) if f.endswith(".lab")][utteranceID]
    except Exception as e:
        raise Exception(f"No .lab file found for speakerID {speakerID} and utteranceID {utteranceID}.") from e
    return os.path.join(JVS_DIR, f"{sample_folder}/lab/mon/{result_file}")

def GetFiles(speakerID, utteranceID):
    assert speakerID < 100
    assert utteranceID < 100
    try:
        wav, lab = GetWavFile(speakerID, utteranceID), GetLabFile(speakerID, utteranceID)
    except IndexError:
        print(f"Folder: {folders[speakerID]}")
        raise ValueError(f"Invalid speakerID {speakerID}/{len(folders)} or utteranceID {utteranceID}/{len([f for f in os.listdir(os.path.join(JVS_DIR, f'{folders[speakerID]}/wav24kHz16bit')) if f.endswith('.wav')])}/{len([f for f in os.listdir(os.path.join(JVS_DIR, f'{folders[speakerID]}/lab/mon')) if f.endswith('.lab')])}.")
    return wav, lab

def Lab2PhonemeSequence(labFilePath):
    with open(labFilePath, 'r') as f:
        phonemes = [line.split()[2] for line in f if line.strip()]
    return phonemes

def PhonemeSequence2IPA(phonemeSequence):
    #1. Replace phonemes with IPA symbols
    #2. Compact same vowel/consonant sequence into geminates
    #3. Decompact my, by, etc... aggregates
    #4. 
    #X. Remove 'sil' and 'pau'
    phonemeSequence = phonemeSequence.copy() 
    
    #print(phonemeSequence)
    
    IPA_MAPPING = {
        'j' : 'dʑ',
        'y' : 'j',
        'u' : 'ɯ',
        "U" : "ɯ̥",
        "I" : "i̥",
        'g' : 'ɡ',
        'r' : 'ɾ',
        'sh' : 'ɕ',
        'ch' : 'tɕ',
        'f' : 'ɸ',
    }
    
    phonemeSequence = [IPA_MAPPING.get(p, p) for p in phonemeSequence]
    
    #Decompact aggregates
    AGGREGATES = {
        "py" : ["p", "j"],
        "ky" : ["k", "j"],
    }
    result = []
    for phoneme in phonemeSequence:
        if phoneme in AGGREGATES:
            result.extend(AGGREGATES[phoneme])
        else:
            result.append(phoneme)
    phonemeSequence = result
    
    #Compact long vowels
    result = [phonemeSequence[0]] if phonemeSequence else []
    for i in range(1, len(phonemeSequence)):
        if result[-1] == phonemeSequence[i] and phonemeSequence[i] in ['n', 'm']: #['a', 'i', 'ɯ', 'e', 'o', 'n']:
            result[-1] += "ː" 
        else:
            result.append(phonemeSequence[i])
            
    phonemeSequence = result 
    
    #Compact geminates
    result = [phonemeSequence[0]] if phonemeSequence else []
    for i in range(1, len(phonemeSequence)):
        if result[-1] == 'cl':
            result[-1] = f"{phonemeSequence[i]}ː"
        else:
            result.append(phonemeSequence[i])
    
    phonemeSequence = result
    
    #Decompact aggregates
    result = [phonemeSequence[0]] if phonemeSequence else []
    for i in range(1, len(phonemeSequence)):
        if len(phonemeSequence[i]) >= 2 and phonemeSequence[i][-1] == "y":
            result.append(phonemeSequence[i][:-1])
            result.append("j") 
        else:
            result.append(phonemeSequence[i])
    
    phonemeSequence = result
    
    #Remove 'sil' and 'pau'
    phonemeSequence = [p for p in phonemeSequence if p not in ['sil', 'pau']]
    
    
    #Adjust phoneme cooarticulation
    result = [phonemeSequence[0]] if phonemeSequence else []
    for i in range(1, len(phonemeSequence)):
        #Palatization
        if phonemeSequence[i] == "j" and result[-1] == "h":
            result[-1] = "ç"

        elif phonemeSequence[i] == "j" and result[-1] == "k":
            result[-1] = "kʲ"

        elif phonemeSequence[i] == "j" and result[-1] == "ɡ":
            result[-1] = "ɡʲ"

        #Nasal assimilation
        elif phonemeSequence[i] == "m" and result[-1] == "N":
            result[-1] = "mː"
            continue
        elif phonemeSequence[i] in ["k", "ɡ"] and result[-1] == "N":
            result[-1] = "ŋ"
        elif phonemeSequence[i] in ["b", "p"] and result[-1] == "N":
            result[-1] = "m"
        elif i == len(phonemeSequence) - 1 and phonemeSequence[i] == "N":
            phonemeSequence[i] = "ɴ"
        elif result[-1] == "N":
            result[-1] = "n"
          
            
        result.append(phonemeSequence[i])
    
    return result

def Lab2IPA(labFilePath):
    phonemeSequence = Lab2PhonemeSequence(labFilePath)
    ipaSequence = PhonemeSequence2IPA(phonemeSequence)
    return " ".join(ipaSequence)


print(Lab2IPA(GetLabFile(2, 21)))

print ({symbol for seq in [Lab2IPA(GetLabFile(0, i)) for i in range(98)] for symbol in seq.split()})

98 Folders found:
['jvs001', 'jvs002', 'jvs003', 'jvs004', 'jvs005', 'jvs007', 'jvs008', 'jvs009', 'jvs010', 'jvs011', 'jvs012', 'jvs013', 'jvs014', 'jvs015', 'jvs016', 'jvs017', 'jvs018', 'jvs019', 'jvs020', 'jvs021', 'jvs022', 'jvs023', 'jvs024', 'jvs025', 'jvs026', 'jvs027', 'jvs029', 'jvs030', 'jvs031', 'jvs032', 'jvs033', 'jvs034', 'jvs035', 'jvs036', 'jvs037', 'jvs038', 'jvs039', 'jvs040', 'jvs041', 'jvs042', 'jvs043', 'jvs044', 'jvs045', 'jvs046', 'jvs047', 'jvs048', 'jvs049', 'jvs050', 'jvs051', 'jvs052', 'jvs053', 'jvs054', 'jvs055', 'jvs056', 'jvs057', 'jvs058', 'jvs059', 'jvs060', 'jvs061', 'jvs062', 'jvs063', 'jvs064', 'jvs065', 'jvs066', 'jvs067', 'jvs068', 'jvs069', 'jvs070', 'jvs071', 'jvs072', 'jvs073', 'jvs074', 'jvs075', 'jvs076', 'jvs077', 'jvs078', 'jvs079', 'jvs080', 'jvs081', 'jvs082', 'jvs083', 'jvs084', 'jvs085', 'jvs086', 'jvs087', 'jvs088', 'jvs089', 'jvs090', 'jvs091', 'jvs092', 'jvs093', 'jvs094', 'jvs095', 'jvs096', 'jvs097', 'jvs098', 'jvs099', 'jvs100']



In [4]:
for utterance in range(100):
    speakerID = 0
    print(Lab2IPA(GetLabFile(speakerID, utterance)))
    

m a t a t o o dʑ i n o j o o n i ɡ o d a i m j o o o o t o j o b a ɾ e ɾ ɯ ɕ ɯ j o o n a m j o o o o n o tɕ ɯ ɯ o o n i h a i s a ɾ e ɾ ɯ k o t o m o o o i
n j ɯ ɯ i ŋ ɡ ɯ ɾ a n d o ɸ ɯ ɯ w a g j ɯ ɯ n j ɯ ɯ o b e e s ɯ̥ t o ɕ i̥ t a ɕ i ɾ o i k ɯ ɾ i i m ɯ s ɯ ɯ p ɯ d e a ɾ i b o s ɯ̥ t o ŋ k ɯ ɾ a m ɯ tɕ a ɯ d a a t o m o j o b a ɾ e ɾ ɯ
k o m p j ɯ ɯ t a ɡ e e m ɯ n o m e e k a a j a g j o o k a i d a n t a i n a d o n i k a n ɾ e n s ɯ ɾ ɯ dʑ i m b ɯ ts ɯ n o k a t e ɡ o ɾ i
s a a b i s ɯ m a n e e dʑ a a d o o n j ɯ ɯ e k i n o t a m e o o i m a tɕ i e k i̥ k a ɾ a e ŋ k a k ɯ̥ k a n ɾ i ɕ i̥ t e i ɾ ɯ
ɕ i ɾ ɯ b a a s a a ɸ a a ɕ ɯ ɯ ɡ e k i dʑ i k e mː a d e n i ɾ i tɕ a a z ɯ w a tɕ i i m ɯ m e e t o t o m o n i k o k ɯ̥ s a i t e k i n i s ɯ ɯ p a a h i i ɾ o o o j o b i j ɯ ɯ m e e dʑ i n t o ɕ i̥ t e n i n tɕ i̥ s a ɾ e t e i ɾ ɯ
ts ɯ j ɯ ɾ e n h a ɾ ɯ t o r j o o w a b j ɯ ɾ ɯ t e m b e ɾ ɯ k ɯ r j o o n i h e n n j ɯ ɯ s a ɾ e t a
dʑ i k a n r j o o i k i̥ t o k ɯ ɯ k a n r

In [5]:
import resampy
import soundfile as sf
import numpy as np
from scipy.signal import resample_poly
from math import gcd


audio, sr = sf.read(f"{JVS_DIR}/jvs001/parallel100/wav24kHz16bit/VOICEACTRESS100_001.wav", dtype="float32")
print(f"Original sample rate: {sr} Hz")
print(f"Original bit depth: {audio.dtype}")

def ResampleAudio(audio, sr):
    g = gcd(sr, 16000)
    audio = resample_poly(audio, up=16000 // g, down=sr // g, axis=0)
    return audio

Original sample rate: 24000 Hz
Original bit depth: float32


In [6]:
import pandas as pd
import numpy as np
import random
from tqdm import tqdm

def BuildJVSRecord(speakerID, utteranceID):
    wavPath, labPath = GetFiles(speakerID, utteranceID)

    audio, sr = sf.read(wavPath, dtype="float32")
    audio = ResampleAudio(audio, sr)

    ipa = Lab2IPA(labPath)

    return {
        "audio": {"array": audio.astype(np.float32), "sampling_rate": 16000, "path": wavPath},
        "IPA": ipa,
        "speakerIDs": speakerID,
        
    }

records = [
    BuildJVSRecord(speakerID, utteranceID)
    for speakerID in tqdm(range(98), desc="Building records")
    for utteranceID in range(98)
]

print(f"Built {len(records)} records")


Building records: 100%|██████████| 98/98 [11:00<00:00,  6.74s/it]

Built 9604 records


In [7]:
TRAIN_SPEAKERS, VAL_SPEAKERS, TEST_SPEAKERS = 80, 10, 10

speakerIDs = list(range(98))
random.Random(42).shuffle(speakerIDs)

trainSpeakers = set(speakerIDs[:80])
valSpeakers = set(speakerIDs[80:89])
testSpeakers = set(speakerIDs[89:])

splitRecords = {"train": [], "validation": [], "test": []}

for speakerID in speakerIDs:
    if speakerID in trainSpeakers:
        splitRecords["train"].extend([r for r in records if r["speakerIDs"] == speakerID])
    elif speakerID in valSpeakers:
        splitRecords["validation"].extend([r for r in records if r["speakerIDs"] == speakerID])
    elif speakerID in testSpeakers:
        splitRecords["test"].extend([r for r in records if r["speakerIDs"] == speakerID])

print("=== Split Sizes (by speaker, held out) ===")
for split, recs in splitRecords.items():
    print(f"{split}: {len(recs)} utterances")


=== Split Sizes (by speaker, held out) ===
train: 7840 utterances
validation: 882 utterances
test: 882 utterances


In [8]:
!python -m pip install datasets


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pickle

CHECKPOINT_PATH = "Data/splitRecords_checkpoint.pkl"
os.makedirs("Data", exist_ok=True)

# Save checkpoint before memory-intensive operation
with open(CHECKPOINT_PATH, 'wb') as f:
    pickle.dump(splitRecords, f)
    print(f"Checkpoint saved to {CHECKPOINT_PATH}")
    print(f"  train: {len(splitRecords['train'])} records")
    print(f"  validation: {len(splitRecords['validation'])} records")
    print(f"  test: {len(splitRecords['test'])} records")

Checkpoint saved to Data/splitRecords_checkpoint.pkl
  train: 7840 records
  validation: 882 records
  test: 882 records


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [4]:
import pickle
import os
import gc
from io import BytesIO

import numpy as np
import soundfile as sf
from datasets import Dataset, Features, Audio, Value

CHECKPOINT_PATH = "drive/MyDrive/Tsukuba/Datasets and Models/splitRecords_checkpoint.pkl"
PARQUET_DIR = "drive/MyDrive/Tsukuba/Datasets and Models"
os.makedirs(PARQUET_DIR, exist_ok=True)

# Audio stored as pre-encoded bytes; matches the English parquet schema (audio, IPA).
FEATURES = Features({"audio": Audio(sampling_rate=16000), "IPA": Value("string")})

def EncodeToWavBytes(array, sr=16000):
    a = np.asarray(array, dtype=np.float32)
    buffer = BytesIO()
    sf.write(buffer, a, sr, format="wav")
    return buffer.getvalue()

with open(CHECKPOINT_PATH, "rb") as f:
    splitRecords = pickle.load(f)

for split in ("train", "validation", "test"):
    recs = splitRecords.pop(split)  # drop from dict so its memory can be reclaimed
    print(f"\nProcessing {split} ({len(recs)} records)...")

    # Encode into WAV bytes, freeing each raw float32 array as we consume it.
    encoded = {"audio": [], "IPA": []}
    while recs:
        r = recs.pop()  # pop from the end -> list shrinks, arrays get freed
        encoded["audio"].append({"bytes": EncodeToWavBytes(r["audio"]["array"]), "path": None})
        encoded["IPA"].append(r["IPA"])
    del recs
    gc.collect()

    splitDs = Dataset.from_dict(encoded, features=FEATURES)
    del encoded
    gc.collect()

    out_path = os.path.join(PARQUET_DIR, f"JP_{split}.parquet")
    splitDs.to_parquet(out_path)
    print(f"✓ Saved {split}: {len(splitDs)} rows -> {out_path}")

    del splitDs
    gc.collect()


Processing train (7840 records)...


Creating parquet from Arrow format:   0%|          | 0/79 [00:00<?, ?ba/s]

✓ Saved train: 7840 rows -> drive/MyDrive/Tsukuba/Datasets and Models/JP_train.parquet

Processing validation (882 records)...


Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

✓ Saved validation: 882 rows -> drive/MyDrive/Tsukuba/Datasets and Models/JP_validation.parquet

Processing test (882 records)...


Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

✓ Saved test: 882 rows -> drive/MyDrive/Tsukuba/Datasets and Models/JP_test.parquet
